# FNO 4D Space-Time Manual Validation
This notebook loads a single batch from the GINO FEFLOW dataset and executes a manual forward pass and loss computation using the 4D Space-Time FNO architecture, replicating the configuration from `hpc/configs/fno/forcing_4d.sh`.


In [1]:
import sys
import os
import torch

# Ensure we can import from src
sys.path.append(os.path.abspath('/scratch/yl75/ak4177/src/GW_SciML/'))

from src.data.patch_dataset_multi_col import GWPatchDatasetMultiCol
from src.data.data_utils import (
    calculate_coord_transform,
    calculate_forcings_transform,
    calculate_obs_transform,
    create_patch_datasets,
    make_collate_fn,
)
from src.models.neuralop.fno import FNOInterpolate
from src.models.neuralop.losses import variance_aware_multicol_loss
from src.training import configure_target_col_indices

# For DataParallel formatting compatibility (unwrap)
from src.training.parallel_utils import broadcast_static_inputs_for_dp



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


### 1. Configuration Setup
Setup the parameters exactly as specified in the `forcing_4d.sh` config.


In [2]:
# Configuration aligned with forcing_4d.sh
BASE_DATA_DIR = '/scratch/yl75/ak4177/data/feflow_data/'

args_dict = {
    'base_data_dir': BASE_DATA_DIR,
    'raw_data_dir': os.path.join(BASE_DATA_DIR, 'all'),
    'patch_data_dir': os.path.join(BASE_DATA_DIR, 'patch_all_ts'),
    'target_cols': ['mass_concentration', 'head'],
    'batch_size': 5, # Small batch for manual validation
    'input_window_size': 10,
    'output_window_size': 10,
    'train_stride': 5,
    'lambda_conc_focus': 0.5,
    'padding_mode': 'border',
    'sampling_strategy': 'static',
    'resolution_ratio': 0.3,
    'min_resolution_ratio': 0.20,
    'forcings_required': True,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # Model architecture parameters from define_model_parameters
    'coord_dim': 3,
    'fno_n_layers': 4,
    'fno_n_modes': (8, 8, 6, 8),
    'fno_hidden_channels': 128,
    'lifting_channels': 64,
    'projection_channel_ratio': 2,
    'align_corners': False
}

import argparse
args = argparse.Namespace(**args_dict)
args.n_target_cols = len(args.target_cols)
args.in_channels = args.n_target_cols + 2 * 4  # 2 obs + 8 forcings
args.out_channels = args.n_target_cols
args.latent_query_dims = (16, 16, 8, args.input_window_size)
args = configure_target_col_indices(args)

print(f'Device: {args.device}')



Target columns: ['mass_concentration', 'head']
Target column indices: [0, 1]
Device: cuda


In [3]:
from pprint import pprint
pprint(vars(args))

{'align_corners': False,
 'base_data_dir': '/scratch/yl75/ak4177/data/feflow_data/',
 'batch_size': 5,
 'coord_dim': 3,
 'device': 'cuda',
 'fno_hidden_channels': 128,
 'fno_n_layers': 4,
 'fno_n_modes': (8, 8, 6, 8),
 'forcings_required': True,
 'in_channels': 10,
 'input_window_size': 10,
 'lambda_conc_focus': 0.5,
 'latent_query_dims': (16, 16, 8, 10),
 'lifting_channels': 64,
 'min_resolution_ratio': 0.2,
 'n_target_cols': 2,
 'out_channels': 2,
 'output_window_size': 10,
 'padding_mode': 'border',
 'patch_data_dir': '/scratch/yl75/ak4177/data/feflow_data/patch_all_ts',
 'projection_channel_ratio': 2,
 'raw_data_dir': '/scratch/yl75/ak4177/data/feflow_data/all',
 'resolution_ratio': 0.3,
 'sampling_strategy': 'static',
 'target_col_indices': [0, 1],
 'target_cols': ['mass_concentration', 'head'],
 'train_stride': 5}


### 2. Dataset and Transforms
Initialize transforms, dataset, and grab a single batch.


In [4]:
print('Calculating transforms...')
coord_transform = calculate_coord_transform(args.raw_data_dir)
obs_transform = calculate_obs_transform(args.raw_data_dir, target_obs_cols=['mass_concentration', 'head', 'pressure'])
forcings_transform = calculate_forcings_transform()

print('Loading dataset (this might take a moment)...')
train_ds, val_ds = create_patch_datasets(
    dataset_class=GWPatchDatasetMultiCol,
    patch_data_dir=args.patch_data_dir,
    coord_transform=coord_transform,
    obs_transform=obs_transform,
    target_col_indices=args.target_col_indices,
    input_window_size=args.input_window_size,
    output_window_size=args.output_window_size,
    forcings_required=args.forcings_required,
    forcings_transform=forcings_transform,
    resolution_ratio=args.resolution_ratio,
    min_resolution_ratio=args.min_resolution_ratio,
    sampling_strategy=args.sampling_strategy,
    train_stride=args.train_stride,
)

args._time_values = train_ds.time_values
collate_fn = make_collate_fn(args, coord_dim=args.coord_dim)

Calculating transforms...
Coordinate mean: [ 3.57225665e+05  6.45774324e+06 -9.27782248e+00]
Coordinate std: [569.1699999  566.35797379  15.26565618]
Calculating observation transform for columns: ['mass_concentration', 'head', 'pressure']
Output mean: [1.84147516e+04 3.42851163e-01 9.43270494e+01]
Output std: [1.51167586e+04 2.34089747e-01 1.51369751e+02]
Loading dataset (this might take a moment)...
Loaded time_values.npy: 1909 timesteps, range [0.00, 908.38] days
Static sampling enabled. Uniform patch ratio: 0.3000
Patch 1: subsampling applied (ratio=0.3000). Core nodes 3409 -> 1137, Ghost nodes 170 -> 51
Patch 2: subsampling applied (ratio=0.3000). Core nodes 3332 -> 1111, Ghost nodes 332 -> 99
Patch 3: subsampling applied (ratio=0.3000). Core nodes 4312 -> 1438, Ghost nodes 215 -> 64
Patch 4: subsampling applied (ratio=0.3000). Core nodes 2905 -> 969, Ghost nodes 435 -> 130
Patch 5: subsampling applied (ratio=0.3000). Core nodes 2562 -> 854, Ghost nodes 128 -> 38
Patch 6: subsampl

In [5]:
# Grab a single batch manually
print('Preparing single batch...')
raw_batch = [val_ds[i] for i in range(args.batch_size)]
batch = collate_fn(raw_batch)

print(f"Batch keys: {list(batch.keys())}")
print(f"Input X shape: {batch['x'].shape}")
print(f"Output Y shape: {batch['y'].shape}")

Preparing single batch...
Batch keys: ['patch_id', 'input_coords', 'output_coords', 'latent_queries', 'x', 'y', 'core_len', 'n_pts', 'T_in', 'T_out', 'weights', 'spatial_coords']
Input X shape: torch.Size([5, 11880, 10])
Output Y shape: torch.Size([5, 11880, 2])


In [6]:
raw_batch = [train_ds[i] for i in range(args.batch_size)]
batch = collate_fn(raw_batch)
batch['input_coords'][:, -1]

tensor([-1.7294, -1.7284, -1.7276,  ..., -1.7171, -1.7149, -1.7133])

In [7]:
raw_batch = [val_ds[len(val_ds)-i-1] for i in range(args.batch_size)]
batch = collate_fn(raw_batch)
batch['input_coords'][:, -1]

tensor([1.7074, 1.7087, 1.7114,  ..., 1.7181, 1.7192, 1.7204])

### 3. Model Initialization
Create the FNO 4D Interpolate model.


In [8]:
print('Initializing FNOInterpolate...')
model = FNOInterpolate(
    latent_query_dims=args.latent_query_dims,
    coord_dim=args.coord_dim,
    in_channels=args.in_channels,
    out_channels=args.out_channels,
    lifting_channels=args.lifting_channels,
    projection_channel_ratio=args.projection_channel_ratio,
    fno_n_modes=args.fno_n_modes,
    fno_hidden_channels=args.fno_hidden_channels,
    fno_n_layers=args.fno_n_layers,
    align_corners=args.align_corners,
    padding_mode=args.padding_mode,
).to(args.device)

model.eval()  # Put model in eval mode for deterministic forward pass
print(f'Model initialized with {sum(p.numel() for p in model.parameters())} parameters.')

Initializing FNOInterpolate...
Using Factorise Space-Time Spectral Conv
Model initialized with 34318914 parameters.


### 4. Forward Pass
Perform the 4D space-time forward pass safely using the unwrapped model logic.


In [11]:
# Prepare inputs for the model
input_coords = batch['input_coords'].to(args.device).float()
output_coords = batch['output_coords'].to(args.device).float()
latent_queries = batch['latent_queries'].to(args.device).float()
x = batch['x'].to(args.device).float()
batch_size = x.shape[0]

print('Running forward pass...')
with torch.no_grad():
    # Pass unbatched static coordinates directly via keyword arguments since model is unwrapped
    outputs = model(
        input_geom=input_coords,
        latent_queries=latent_queries,
        output_queries=output_coords,
        x=x
    )

print(f'Model output shape: {outputs.shape}')

Running forward pass...
Model output shape: torch.Size([5, 9780, 2])


### 5. Core Extraction & Loss Computation
Extract core points correctly by reshaping and applying the variance-aware multi-col loss.


In [10]:
from src.data.data_utils import reshape_multi_col_predictions
from src.models.neuralop.losses import LpLoss

y = batch['y'].to(args.device).float()
core_len = batch['core_len']
T_out = batch['T_out']
C_obs = args.n_target_cols

print(f'Core length: {core_len}, T_out: {T_out}, C_obs: {C_obs}')

Core length: 830, T_out: 10, C_obs: 2


In [11]:


# # 3. Permute the channels and coord dimensions
# core_outputs = torch.permute(core_outputs, dims=(0, 3, 1, 2))
# core_targets = torch.permute(core_targets, dims=(0, 3, 1, 2))

# print(f"Permuted outputs: {core_outputs.shape},  targets: {core_targets.shape}")

In [12]:
# 4. Extract weights (safely taking only the spatial nodes)
weights = batch['weights'].to(args.device).float()
# core_weights = weights[::T_out][:core_len]
core_weights = weights.reshape((-1, T_out))[:core_len, 0]

# print(f'Core outputs flat shape: {core_outputs_flat.shape}')
print(f'Core weights shape: {core_weights.shape}')

Core weights shape: torch.Size([830])


In [13]:
# 1. Reshape outputs and targets to separate spatial and temporal dims
outputs_reshaped = reshape_multi_col_predictions(outputs, T_out, C_obs)
y_reshaped = reshape_multi_col_predictions(y, T_out, C_obs)

# 2. Extract core points
core_outputs = outputs_reshaped[:, :core_len, :, :]  # [B, N, T, C]
core_targets = y_reshaped[:, :core_len, :, :]        # [B, N, T, C]

# Apply temporal pushforward weights (e.g. 1.0 to 2.0)
t_weights = torch.linspace(1.0, 2.0, T_out, device=core_outputs.device) # [T]

# 3. Permute to [B, T, N, C]
core_outputs_permute = torch.permute(core_outputs, dims=(0, 2, 1, 3)) 
core_targets_permute = torch.permute(core_targets, dims=(0, 2, 1, 3))

# ==========================================
# 4. Global Loss (Relative L2 over N and C)
# ==========================================
# d=2 flattens N and C together. reduce_dims=0 takes the mean over Batch.
# Output shape of global_loss_fn is [T]
global_loss_fn = LpLoss(d=2, p=2, reduce_dims=0, reductions='mean') 

global_loss_per_t = global_loss_fn(core_outputs_permute, core_targets_permute)
# Weighted average over Time
global_loss = (t_weights * global_loss_per_t).sum() / t_weights.sum()

print(global_loss_per_t.shape, global_loss)

# ==========================================
# 5. Variance-Aware Concentration Loss (Weighted Relative L2)
# ==========================================
conc_idx = args.target_cols.index('mass_concentration')
conc_pred = core_outputs_permute[..., conc_idx]  # [B, T, N]
conc_true = core_targets_permute[..., conc_idx]  # [B, T, N]

# Normalize spatial weights to sum to 1 and take square root
normalized_weights = core_weights / core_weights.sum()
sqrt_weights = torch.sqrt(normalized_weights).view(1, 1, -1)  # [1, 1, N]

# Pre-weight inputs to inject spatial variance weights inside the L2 Norm
weighted_conc_pred = conc_pred * sqrt_weights
weighted_conc_true = conc_true * sqrt_weights

# d=1 flattens N. reduce_dims=0 takes mean over B. Output is [T]. 
# ** IMPORTANT: Make sure you update the LpLoss class to accept and use `eps=1e-8` **
local_loss_fn = LpLoss(d=1, p=2, reduce_dims=0, reductions='mean', eps=1e-8)
local_loss_per_t = local_loss_fn(weighted_conc_pred, weighted_conc_true)

# Weighted average over Time
local_conc_loss = (t_weights * local_loss_per_t).sum() / t_weights.sum()
print(local_loss_per_t.shape, local_conc_loss)

# ==========================================
# 6. Combine losses
# ==========================================
loss = (1 - args.lambda_conc_focus) * global_loss + args.lambda_conc_focus * local_conc_loss
loss, args.lambda_conc_focus

torch.Size([10]) tensor(1.0298, device='cuda:0')
torch.Size([10]) tensor(1.0717, device='cuda:0')


(tensor(1.0508, device='cuda:0'), 0.5)

In [14]:
# 5. Compute actual loss
loss, global_loss, conc_var_loss = variance_aware_multicol_loss(
    core_outputs, 
    core_targets, 
    core_weights,
    output_window_size=args.output_window_size,
    target_cols=args.target_cols,
    lambda_conc_focus=args.lambda_conc_focus,
)

print('-'*40)
print(f'Variance-aware multicol loss: {loss.item():.6f}')
print(f'  - Global loss: {global_loss.item():.6f}')
print(f'  - Concentration var loss: {conc_var_loss.item():.6f}')

----------------------------------------
Variance-aware multicol loss: 1.050766
  - Global loss: 1.029795
  - Concentration var loss: 1.071736
